# First kernels: threads, grids, images and matrices (chapter 3)

Session 11 (`11_intro_gpu_matrix`): a GPU gives every element its own thread. This folder writes the first kernels of the course, from a kernel that only says hello to the matrix product with one thread per output element. Everything here is index arithmetic: each thread turns its coordinates (`blockIdx`, `blockDim`, `threadIdx`) into the position of the data it owns, and checks that the position exists.

This notebook goes **step by step**: 12 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/01_first_kernels/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 12: `01_hello_threads.cu`

The first kernel: a grid of 2 blocks of 4 threads, each thread prints its `blockIdx.x`, `threadIdx.x` and global index `blockIdx.x * blockDim.x + threadIdx.x` with `printf`, and writes that index to device memory so the host can check it. The order of the printed lines is not fixed.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_hello_threads.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int BLOCKS = 2;
const int THREADS_PER_BLOCK = 4;
const int TOTAL_THREADS = BLOCKS * THREADS_PER_BLOCK;

__global__ void helloKernel(int* globalIndexOf) {
  int block = blockIdx.x;
  int thread = threadIdx.x;
  int i = block * blockDim.x + thread;

  printf("hello from block %d, thread %d: global index %d\n", block, thread, i);
  globalIndexOf[i] = i;
}

static std::vector<int> runHello() {
  const size_t bytes = TOTAL_THREADS * sizeof(int);
  std::vector<int> globalIndexOf(TOTAL_THREADS, -1);

  int* globalIndexOf_d;
  cudaMalloc(&globalIndexOf_d, bytes);

  launch(helloKernel, dim3(BLOCKS), dim3(THREADS_PER_BLOCK), globalIndexOf_d);

  cudaMemcpy(globalIndexOf.data(), globalIndexOf_d, bytes, cudaMemcpyDeviceToHost);
  cudaFree(globalIndexOf_d);
  return globalIndexOf;
}

static bool everyThreadWroteItsIndex(const std::vector<int>& globalIndexOf) {
  for (int i = 0; i < TOTAL_THREADS; ++i) {
    if (globalIndexOf[i] != i) {
      return false;
    }
  }
  return true;
}

int main() {
  std::printf("backend: %s\n", CUDA_BACKEND);
  std::printf(
      "grid: %d blocks of %d threads = %d threads\n\n", BLOCKS, THREADS_PER_BLOCK, TOTAL_THREADS);

  const std::vector<int> globalIndexOf = runHello();

  std::printf("\nglobal index written by each thread:");
  for (int index : globalIndexOf) {
    std::printf(" %d", index);
  }
  std::printf("\n");

  const bool ok = everyThreadWroteItsIndex(globalIndexOf);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_hello_threads.cu -o 01_hello_threads && ./01_hello_threads

**What to look at in the output:** Each line names a block, a thread and the global index `block * 4 + thread`: block 1, thread 3 has index 7. The lines may come out in any order: on the GPU the four lines of one block (one warp) usually stay together, but which block prints first can change between runs. Still, the array copied back is always 0 1 2 3 4 5 6 7, because each thread writes to its own slot. Try: set `BLOCKS = 3` and the indices run to 11.

## Step 2 of 12: `02_vector_add.cu`

Vector addition with the host code of the slides: `cudaMalloc` three arrays, `cudaMemcpy` A and B to the device, launch one block of 8 threads (thread i adds element i), copy C back, `cudaFree`.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_vector_add.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int N = 8;

__global__ void vecAddKernel(const float* A, const float* B, float* C) {
  int i = threadIdx.x;
  C[i] = A[i] + B[i];
}

static void vecAdd(const float* A, const float* B, float* C, int n) {
  const size_t bytes = n * sizeof(float);

  float* A_d;
  float* B_d;
  float* C_d;
  cudaMalloc(&A_d, bytes);
  cudaMalloc(&B_d, bytes);
  cudaMalloc(&C_d, bytes);

  cudaMemcpy(A_d, A, bytes, cudaMemcpyHostToDevice);
  cudaMemcpy(B_d, B, bytes, cudaMemcpyHostToDevice);

  launch(vecAddKernel, dim3(1), dim3(n), A_d, B_d, C_d);

  cudaMemcpy(C, C_d, bytes, cudaMemcpyDeviceToHost);

  cudaFree(A_d);
  cudaFree(B_d);
  cudaFree(C_d);
}

static void vecAddOnCpu(const float* A, const float* B, float* C, int n) {
  for (int i = 0; i < n; ++i) {
    C[i] = A[i] + B[i];
  }
}

static void printVector(const char* label, const std::vector<float>& v) {
  std::printf("%s =", label);
  for (float x : v) {
    std::printf(" %5.1f", x);
  }
  std::printf("\n");
}

int main() {
  std::vector<float> A(N);
  std::vector<float> B(N);
  for (int i = 0; i < N; ++i) {
    A[i] = static_cast<float>(i);
    B[i] = 10.0f * (i + 1);
  }

  std::vector<float> C(N, 0.0f);
  vecAdd(A.data(), B.data(), C.data(), N);

  std::vector<float> expected(N);
  vecAddOnCpu(A.data(), B.data(), expected.data(), N);

  std::printf("one block of %d threads, thread i adds element i\n", N);
  printVector("A", A);
  printVector("B", B);
  printVector("C", C);

  const bool ok = C == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_vector_add.cu -o 02_vector_add && ./02_vector_add

**What to look at in the output:** Read the three lines column by column: column i is the work of thread i, for example C[3] = 3 + 40 = 43. All 8 threads are in one block and no thread uses another thread's element.

## Step 3 of 12: `03_vector_add_any_n.cu`

`vecAddKernel` for any n: several blocks, the global index, the guard `if (i < n)` and the grid size by ceiling division `(n + 3) / 4`. With n = 10 and blocks of 4, 12 threads run and 2 stop at the guard. (slide: *`dim3`: describing the grid and the blocks*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_vector_add_any_n.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int N = 10;
const int BLOCK_SIZE = 4;

__global__ void vecAddKernel(const float* A, const float* B, float* C, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    C[i] = A[i] + B[i];
  }
}

static int blocksToCover(int n, int blockSize) {
  return (n + blockSize - 1) / blockSize;
}

static void vecAdd(const float* A, const float* B, float* C, int n) {
  const size_t bytes = n * sizeof(float);

  float* A_d;
  float* B_d;
  float* C_d;
  cudaMalloc(&A_d, bytes);
  cudaMalloc(&B_d, bytes);
  cudaMalloc(&C_d, bytes);

  cudaMemcpy(A_d, A, bytes, cudaMemcpyHostToDevice);
  cudaMemcpy(B_d, B, bytes, cudaMemcpyHostToDevice);

  dim3 block(BLOCK_SIZE);
  dim3 grid(blocksToCover(n, BLOCK_SIZE));
  launch(vecAddKernel, grid, block, A_d, B_d, C_d, n);

  cudaMemcpy(C, C_d, bytes, cudaMemcpyDeviceToHost);

  cudaFree(A_d);
  cudaFree(B_d);
  cudaFree(C_d);
}

static void vecAddOnCpu(const float* A, const float* B, float* C, int n) {
  for (int i = 0; i < n; ++i) {
    C[i] = A[i] + B[i];
  }
}

static void printVector(const char* label, const std::vector<float>& v) {
  std::printf("%s =", label);
  for (float x : v) {
    std::printf(" %5.1f", x);
  }
  std::printf("\n");
}

int main() {
  std::vector<float> A(N);
  std::vector<float> B(N);
  for (int i = 0; i < N; ++i) {
    A[i] = static_cast<float>(i);
    B[i] = 10.0f * (i + 1);
  }

  std::vector<float> C(N, 0.0f);
  vecAdd(A.data(), B.data(), C.data(), N);

  std::vector<float> expected(N);
  vecAddOnCpu(A.data(), B.data(), expected.data(), N);

  const int blocks = blocksToCover(N, BLOCK_SIZE);
  const int launched = blocks * BLOCK_SIZE;
  std::printf("n = %d, %d threads per block: (%d + %d) / %d = %d blocks\n",
              N,
              BLOCK_SIZE,
              N,
              BLOCK_SIZE - 1,
              BLOCK_SIZE,
              blocks);
  std::printf("%d threads launched, %d stop at the guard i < n\n", launched, launched - N);
  printVector("A", A);
  printVector("B", B);
  printVector("C", C);

  const bool ok = C == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_vector_add_any_n.cu -o 03_vector_add_any_n && ./03_vector_add_any_n

**What to look at in the output:** The first two lines are the grid arithmetic: (10 + 3) / 4 = 3 blocks, 12 threads, and threads 10 and 11 (the last two of block 2) stop at `i < n`, so C still has exactly 10 values, ending in 9 + 100 = 109. Try: set `N = 13`; you get 4 blocks, 16 threads and 3 stop at the guard.

## Step 4 of 12: `04_grid_2d.cu`

A 2D grid over a 6×5 picture: `dim3 block(4, 4)`, a 2×2 grid, `col` from x and `row` from y, the 2D guard and the row-major index `row * width + col`. Prints which block owns each pixel: 64 threads for 30 pixels. (slides: *Launching a 2D grid*, *Flattening by hand*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_grid_2d.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int WIDTH = 6;
const int HEIGHT = 5;
const int PIXELS = WIDTH * HEIGHT;
const int BLOCK_SIDE = 4;

__global__ void mapPixels(int* blockOf, int* rowOf, int* colOf, int width, int height) {
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  int row = blockIdx.y * blockDim.y + threadIdx.y;

  if (col < width && row < height) {
    int flat = row * width + col;
    blockOf[flat] = blockIdx.y * gridDim.x + blockIdx.x;
    rowOf[flat] = row;
    colOf[flat] = col;
  }
}

struct PixelOwners {
  std::vector<int> blockOf = std::vector<int>(PIXELS, -1);
  std::vector<int> rowOf = std::vector<int>(PIXELS, -1);
  std::vector<int> colOf = std::vector<int>(PIXELS, -1);
};

static int blocksToCover(int n, int blockSide) {
  return (n + blockSide - 1) / blockSide;
}

static PixelOwners mapPixelsOnDevice(dim3 grid, dim3 block) {
  const size_t bytes = PIXELS * sizeof(int);
  PixelOwners owners;

  int* blockOf_d;
  int* rowOf_d;
  int* colOf_d;
  cudaMalloc(&blockOf_d, bytes);
  cudaMalloc(&rowOf_d, bytes);
  cudaMalloc(&colOf_d, bytes);

  launch(mapPixels, grid, block, blockOf_d, rowOf_d, colOf_d, WIDTH, HEIGHT);

  cudaMemcpy(owners.blockOf.data(), blockOf_d, bytes, cudaMemcpyDeviceToHost);
  cudaMemcpy(owners.rowOf.data(), rowOf_d, bytes, cudaMemcpyDeviceToHost);
  cudaMemcpy(owners.colOf.data(), colOf_d, bytes, cudaMemcpyDeviceToHost);

  cudaFree(blockOf_d);
  cudaFree(rowOf_d);
  cudaFree(colOf_d);
  return owners;
}

static void printMap(const char* title, const std::vector<int>& valueOf) {
  std::printf("%s\n", title);
  for (int row = 0; row < HEIGHT; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      std::printf(" %3d", valueOf[row * WIDTH + col]);
    }
    std::printf("\n");
  }
}

static bool ownersAreCorrect(const PixelOwners& owners, int blocksAcross) {
  for (int flat = 0; flat < PIXELS; ++flat) {
    const int row = flat / WIDTH;
    const int col = flat % WIDTH;
    const int block = (row / BLOCK_SIDE) * blocksAcross + col / BLOCK_SIDE;

    if (owners.rowOf[flat] != row || owners.colOf[flat] != col || owners.blockOf[flat] != block) {
      return false;
    }
  }
  return true;
}

int main() {
  dim3 block(BLOCK_SIDE, BLOCK_SIDE);
  dim3 grid(blocksToCover(WIDTH, BLOCK_SIDE), blocksToCover(HEIGHT, BLOCK_SIDE));
  const int launched = grid.x * grid.y * block.x * block.y;

  std::printf("picture %d x %d, blocks of %d x %d, grid %u x %u blocks\n",
              WIDTH,
              HEIGHT,
              BLOCK_SIDE,
              BLOCK_SIDE,
              grid.x,
              grid.y);
  std::printf("%d threads launched for %d pixels\n\n", launched, PIXELS);

  const PixelOwners owners = mapPixelsOnDevice(grid, block);

  std::vector<int> flatIndex(PIXELS);
  for (int flat = 0; flat < PIXELS; ++flat) {
    flatIndex[flat] = owners.rowOf[flat] * WIDTH + owners.colOf[flat];
  }

  printMap("block that owns each pixel (blockIdx.y * gridDim.x + blockIdx.x):", owners.blockOf);
  printMap("\nrow-major index row * width + col:", flatIndex);

  const bool ok = ownersAreCorrect(owners, grid.x);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_grid_2d.cu -o 04_grid_2d && ./04_grid_2d

**What to look at in the output:** The first map shows the four blocks as 4×4 squares: block 0 covers columns 0 to 3 of rows 0 to 3, while blocks 1, 2 and 3 are cut by the picture's edge (block 3 owns only pixels (4, 4) and (4, 5)). The second map is the row-major index: pixel (row 2, col 3) is 2 × 6 + 3 = 15. 34 of the 64 threads fail the 2D guard.

## Step 5 of 12: `05_color_to_gray.cu`

`colorToGray` on a 4×3 RGB picture: each thread reads 3 bytes at `rgbOffset = grayOffset * CHANNELS` and writes 1 gray byte. A pure map: no thread touches another thread's pixel. (slides: *Launching a 2D grid*, *The grayscale kernel*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_color_to_gray.cu
#include "cuda_compat.h"

#include <cstdio>
#include <cstdlib>
#include <vector>

#define CHANNELS 3

const int WIDTH = 4;
const int HEIGHT = 3;
const int PIXELS = WIDTH * HEIGHT;
const int BLOCK_SIDE = 16;
const int ROUNDING_TOLERANCE = 1;

using Bytes = std::vector<unsigned char>;

__global__ void colorToGray(unsigned char* gray, const unsigned char* rgb, int width, int height) {
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  int row = blockIdx.y * blockDim.y + threadIdx.y;

  if (col < width && row < height) {
    int grayOffset = row * width + col;
    int rgbOffset = grayOffset * CHANNELS;

    unsigned char r = rgb[rgbOffset];
    unsigned char g = rgb[rgbOffset + 1];
    unsigned char b = rgb[rgbOffset + 2];
    gray[grayOffset] = (unsigned char)(0.299f * r + 0.587f * g + 0.114f * b);
  }
}

static Bytes toGray(const Bytes& rgb, int width, int height) {
  const size_t grayBytes = width * height;
  const size_t rgbBytes = grayBytes * CHANNELS;
  Bytes gray(grayBytes, 0);

  unsigned char* rgb_d;
  unsigned char* gray_d;
  cudaMalloc(&rgb_d, rgbBytes);
  cudaMalloc(&gray_d, grayBytes);
  cudaMemcpy(rgb_d, rgb.data(), rgbBytes, cudaMemcpyHostToDevice);

  dim3 block(BLOCK_SIDE, BLOCK_SIDE);
  dim3 grid((width + BLOCK_SIDE - 1) / BLOCK_SIDE, (height + BLOCK_SIDE - 1) / BLOCK_SIDE);
  launch(colorToGray, grid, block, gray_d, rgb_d, width, height);

  cudaMemcpy(gray.data(), gray_d, grayBytes, cudaMemcpyDeviceToHost);
  cudaFree(rgb_d);
  cudaFree(gray_d);
  return gray;
}

static Bytes toGrayOnCpu(const Bytes& rgb) {
  Bytes gray(PIXELS);
  for (int pixel = 0; pixel < PIXELS; ++pixel) {
    const unsigned char r = rgb[pixel * CHANNELS];
    const unsigned char g = rgb[pixel * CHANNELS + 1];
    const unsigned char b = rgb[pixel * CHANNELS + 2];
    gray[pixel] = (unsigned char)(0.299f * r + 0.587f * g + 0.114f * b);
  }
  return gray;
}

const unsigned char PICTURE[PIXELS][CHANNELS] = {
    {255, 0, 0},
    {0, 255, 0},
    {0, 0, 255},
    {255, 255, 255},
    {255, 255, 0},
    {0, 255, 255},
    {255, 0, 255},
    {0, 0, 0},
    {128, 128, 128},
    {255, 165, 0},
    {128, 0, 128},
    {165, 42, 42},
};

static Bytes tinyPicture() {
  const unsigned char* first = &PICTURE[0][0];
  return Bytes(first, first + PIXELS * CHANNELS);
}

static void printPictures(const Bytes& rgb, const Bytes& gray) {
  for (int row = 0; row < HEIGHT; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      const int pixel = row * WIDTH + col;
      std::printf(" (%3d,%3d,%3d)",
                  rgb[pixel * CHANNELS],
                  rgb[pixel * CHANNELS + 1],
                  rgb[pixel * CHANNELS + 2]);
    }
    std::printf("   ->");
    for (int col = 0; col < WIDTH; ++col) {
      std::printf(" %3d", gray[row * WIDTH + col]);
    }
    std::printf("\n");
  }
}

static bool closeEnough(const Bytes& gray, const Bytes& expected) {
  for (int pixel = 0; pixel < PIXELS; ++pixel) {
    if (std::abs(gray[pixel] - expected[pixel]) > ROUNDING_TOLERANCE) {
      return false;
    }
  }
  return true;
}

int main() {
  const Bytes rgb = tinyPicture();
  const Bytes gray = toGray(rgb, WIDTH, HEIGHT);
  const Bytes expected = toGrayOnCpu(rgb);

  std::printf("picture %d x %d, (r,g,b) on the left, gray on the right\n", WIDTH, HEIGHT);
  printPictures(rgb, gray);

  const bool ok = closeEnough(gray, expected);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_color_to_gray.cu -o 05_color_to_gray && ./05_color_to_gray

**What to look at in the output:** Each gray value is 0.299 r + 0.587 g + 0.114 b, truncated: pure red gives 76, pure green 149 and pure blue only 29, because the eye is most sensitive to green. White stays 255 and black 0.

## Step 6 of 12: `06_blur.cu`

`blurKernel` with `BLUR_SIZE = 1` on a 6×6 picture: every output averages its 3×3 patch, skipping neighbours outside the picture, so corners divide by 4, edges by 6 and the inside by 9. (slides: *The blur kernel*, *Session 10 on the CPU, session 11 on the GPU*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 06_blur.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

#define BLUR_SIZE 1

const int WIDTH = 6;
const int HEIGHT = 6;
const int PIXELS = WIDTH * HEIGHT;
const int BLOCK_SIDE = 16;
const unsigned char DARK = 10;
const unsigned char BRIGHT = 100;

using Bytes = std::vector<unsigned char>;

__global__ void blurKernel(const unsigned char* in, unsigned char* out, int w, int h) {
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  int row = blockIdx.y * blockDim.y + threadIdx.y;

  if (col < w && row < h) {
    int sum = 0;
    int pixels = 0;

    for (int dr = -BLUR_SIZE; dr <= BLUR_SIZE; ++dr) {
      for (int dc = -BLUR_SIZE; dc <= BLUR_SIZE; ++dc) {
        int r = row + dr;
        int c = col + dc;
        if (r >= 0 && r < h && c >= 0 && c < w) {
          sum += in[r * w + c];
          ++pixels;
        }
      }
    }

    out[row * w + col] = (unsigned char)(sum / pixels);
  }
}

static Bytes blur(const Bytes& in, int w, int h) {
  const size_t bytes = w * h;
  Bytes out(bytes, 0);

  unsigned char* in_d;
  unsigned char* out_d;
  cudaMalloc(&in_d, bytes);
  cudaMalloc(&out_d, bytes);
  cudaMemcpy(in_d, in.data(), bytes, cudaMemcpyHostToDevice);

  dim3 block(BLOCK_SIDE, BLOCK_SIDE);
  dim3 grid((w + BLOCK_SIDE - 1) / BLOCK_SIDE, (h + BLOCK_SIDE - 1) / BLOCK_SIDE);
  launch(blurKernel, grid, block, in_d, out_d, w, h);

  cudaMemcpy(out.data(), out_d, bytes, cudaMemcpyDeviceToHost);
  cudaFree(in_d);
  cudaFree(out_d);
  return out;
}

static unsigned char patchAverageOnCpu(const Bytes& in, int row, int col) {
  int sum = 0;
  int pixels = 0;
  for (int r = row - BLUR_SIZE; r <= row + BLUR_SIZE; ++r) {
    for (int c = col - BLUR_SIZE; c <= col + BLUR_SIZE; ++c) {
      const bool inside = r >= 0 && r < HEIGHT && c >= 0 && c < WIDTH;
      if (inside) {
        sum += in[r * WIDTH + c];
        ++pixels;
      }
    }
  }
  return (unsigned char)(sum / pixels);
}

static Bytes blurOnCpu(const Bytes& in) {
  Bytes out(PIXELS);
  for (int row = 0; row < HEIGHT; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      out[row * WIDTH + col] = patchAverageOnCpu(in, row, col);
    }
  }
  return out;
}

static Bytes brightSquareAndCorner() {
  Bytes picture(PIXELS, DARK);
  picture[2 * WIDTH + 2] = BRIGHT;
  picture[2 * WIDTH + 3] = BRIGHT;
  picture[3 * WIDTH + 2] = BRIGHT;
  picture[3 * WIDTH + 3] = BRIGHT;
  picture[WIDTH - 1] = BRIGHT;
  return picture;
}

static void printPicture(const char* title, const Bytes& picture) {
  std::printf("%s\n", title);
  for (int row = 0; row < HEIGHT; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      std::printf(" %3d", picture[row * WIDTH + col]);
    }
    std::printf("\n");
  }
}

int main() {
  const Bytes in = brightSquareAndCorner();
  const Bytes out = blur(in, WIDTH, HEIGHT);
  const Bytes expected = blurOnCpu(in);

  std::printf("%d x %d patch (BLUR_SIZE = %d)\n", 2 * BLUR_SIZE + 1, 2 * BLUR_SIZE + 1, BLUR_SIZE);
  printPicture("input:", in);
  printPicture("blurred:", out);

  const bool ok = out == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 06_blur.cu -o 06_blur && ./06_blur

**What to look at in the output:** The bright 2×2 square becomes a soft spot: its pixels drop to 50 (4 × 100 + 5 × 10 = 450, divided by 9) and the ring around it gets 20 or 30. The bright corner pixel at the top right becomes 32 (130 / 4, only 4 pixels in its patch) and its neighbour on the left 25 (150 / 6). Try: set `BLUR_SIZE` to 2 for a 5×5 patch and a wider, flatter spot.

## Step 7 of 12: `07_matmul.cu`

`matMulKernel` on 4×4 matrices with `BLOCK_WIDTH = 2` (4 blocks of 4 threads): thread (row, col) computes the dot product of row `row` of M and column `col` of N. The grid replaces the two outer loops of the CPU version. (slide: *The matrix multiplication kernel and its host code*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 07_matmul.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int WIDTH = 4;
const int ELEMENTS = WIDTH * WIDTH;
const int BLOCK_WIDTH = 2;

using Matrix = std::vector<float>;

__global__ void matMulKernel(const float* M, const float* N, float* P, int width) {
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if (row < width && col < width) {
    float sum = 0;
    for (int k = 0; k < width; ++k) {
      sum += M[row * width + k] * N[k * width + col];
    }
    P[row * width + col] = sum;
  }
}

static void matMul(const float* M, const float* N, float* P, int width) {
  const size_t bytes = (size_t)width * width * sizeof(float);

  float* M_d;
  float* N_d;
  float* P_d;
  cudaMalloc(&M_d, bytes);
  cudaMalloc(&N_d, bytes);
  cudaMalloc(&P_d, bytes);

  cudaMemcpy(M_d, M, bytes, cudaMemcpyHostToDevice);
  cudaMemcpy(N_d, N, bytes, cudaMemcpyHostToDevice);

  dim3 block(BLOCK_WIDTH, BLOCK_WIDTH);
  dim3 grid((width + BLOCK_WIDTH - 1) / BLOCK_WIDTH, (width + BLOCK_WIDTH - 1) / BLOCK_WIDTH);
  launch(matMulKernel, grid, block, M_d, N_d, P_d, width);

  cudaMemcpy(P, P_d, bytes, cudaMemcpyDeviceToHost);

  cudaFree(M_d);
  cudaFree(N_d);
  cudaFree(P_d);
}

static Matrix matMulOnCpu(const Matrix& M, const Matrix& N) {
  Matrix P(ELEMENTS, 0.0f);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      float sum = 0;
      for (int k = 0; k < WIDTH; ++k) {
        sum += M[row * WIDTH + k] * N[k * WIDTH + col];
      }
      P[row * WIDTH + col] = sum;
    }
  }
  return P;
}

static void printMatrix(const char* name, const Matrix& matrix) {
  std::printf("%s =\n", name);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      std::printf(" %6.1f", matrix[row * WIDTH + col]);
    }
    std::printf("\n");
  }
}

int main() {
  Matrix M(ELEMENTS);
  Matrix N(ELEMENTS);
  for (int i = 0; i < ELEMENTS; ++i) {
    M[i] = static_cast<float>(i + 1);
    N[i] = static_cast<float>((i % WIDTH + i / WIDTH) % 3);
  }

  Matrix P(ELEMENTS, 0.0f);
  matMul(M.data(), N.data(), P.data(), WIDTH);
  const Matrix expected = matMulOnCpu(M, N);

  const int blocksPerSide = (WIDTH + BLOCK_WIDTH - 1) / BLOCK_WIDTH;
  std::printf("%d x %d matrices, blocks of %d x %d threads, grid of %d x %d blocks\n",
              WIDTH,
              WIDTH,
              BLOCK_WIDTH,
              BLOCK_WIDTH,
              blocksPerSide,
              blocksPerSide);
  std::printf("thread (row, col) computes P[row][col] = row of M times column of N\n\n");
  printMatrix("M", M);
  printMatrix("N", N);
  printMatrix("P = M x N", P);

  const bool ok = P == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 07_matmul.cu -o 07_matmul && ./07_matmul

**What to look at in the output:** Check one entry by hand: P[0][0] = 1·0 + 2·1 + 3·2 + 4·0 = 8, row 0 of M times column 0 of N. Columns 0 and 3 of P are equal because columns 0 and 3 of N are equal (0, 1, 2, 0). Each of the 16 threads computed one of these entries.

## Step 8 of 12: `08_error_checking.cu`

Error checking. Steps 1 to 7 call `cudaMalloc`, `cudaMemcpy` and the kernel as the slides do, ignoring the result. CUDA never stops a program on an error: every call returns a `cudaError_t`, and a kernel launch returns nothing, so its error must be asked for with `cudaGetLastError()`. This step launches the vector addition with 2048 threads per block (the limit is 1024), first without checking, then asking for the error, and introduces `CHECK(call)`, which prints the error with file and line and stops. From here on every program wraps its CUDA calls in `CHECK`.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 08_error_checking.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int N = 8;
const int TOO_MANY_THREADS_PER_BLOCK = 2048;
const int THREADS_PER_BLOCK = 256;
const float NOT_COMPUTED = -1.0f;

__global__ void vecAddKernel(const float* A, const float* B, float* C, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    C[i] = A[i] + B[i];
  }
}

struct DeviceVectors {
  float* A_d;
  float* B_d;
  float* C_d;
};

static DeviceVectors copyInputsToDevice(const std::vector<float>& A, const std::vector<float>& B) {
  const size_t bytes = A.size() * sizeof(float);
  DeviceVectors device{};
  CHECK(cudaMalloc(&device.A_d, bytes));
  CHECK(cudaMalloc(&device.B_d, bytes));
  CHECK(cudaMalloc(&device.C_d, bytes));
  CHECK(cudaMemcpy(device.A_d, A.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(device.B_d, B.data(), bytes, cudaMemcpyHostToDevice));
  return device;
}

static void fillOutputWithMarker(const DeviceVectors& device) {
  const std::vector<float> marker(N, NOT_COMPUTED);
  CHECK(cudaMemcpy(device.C_d, marker.data(), N * sizeof(float), cudaMemcpyHostToDevice));
}

static std::vector<float> copyOutputToHost(const DeviceVectors& device) {
  std::vector<float> C(N);
  CHECK(cudaMemcpy(C.data(), device.C_d, N * sizeof(float), cudaMemcpyDeviceToHost));
  return C;
}

static void freeDevice(const DeviceVectors& device) {
  CHECK(cudaFree(device.A_d));
  CHECK(cudaFree(device.B_d));
  CHECK(cudaFree(device.C_d));
}

static void printVector(const char* label, const std::vector<float>& v) {
  std::printf("%s =", label);
  for (float x : v) {
    std::printf(" %5.1f", x);
  }
  std::printf("\n");
}

static void launchWithoutChecking(const DeviceVectors& device) {
  std::printf("1) launch with %d threads per block and do not check:\n",
              TOO_MANY_THREADS_PER_BLOCK);
  fillOutputWithMarker(device);
  launch(vecAddKernel,
         dim3(1),
         dim3(TOO_MANY_THREADS_PER_BLOCK),
         device.A_d,
         device.B_d,
         device.C_d,
         N);
  printVector("   C", copyOutputToHost(device));
  std::printf("   the kernel never ran (C still holds the marker %.1f) and nothing said so\n\n",
              NOT_COMPUTED);
}

static void launchAndAskForTheError(const DeviceVectors& device) {
  std::printf("2) the same launch, then ask cudaGetLastError():\n");
  fillOutputWithMarker(device);
  launch(vecAddKernel,
         dim3(1),
         dim3(TOO_MANY_THREADS_PER_BLOCK),
         device.A_d,
         device.B_d,
         device.C_d,
         N);
  const cudaError_t launchError = cudaGetLastError();
  std::printf("   cudaGetLastError() = %d: \"%s\"\n",
              static_cast<int>(launchError),
              cudaGetErrorString(launchError));
  std::printf(
      "   CHECK(cudaGetLastError()) would print that message with file and line, and stop\n\n");
}

static std::vector<float> launchCorrectlyWithCheck(const DeviceVectors& device) {
  const int blocks = cdiv(N, THREADS_PER_BLOCK);
  std::printf("3) a valid launch, %d block of %d threads, checked with CHECK:\n",
              blocks,
              THREADS_PER_BLOCK);
  fillOutputWithMarker(device);
  launch(
      vecAddKernel, dim3(blocks), dim3(THREADS_PER_BLOCK), device.A_d, device.B_d, device.C_d, N);
  CHECK(cudaGetLastError());
  CHECK(cudaDeviceSynchronize());
  std::vector<float> C = copyOutputToHost(device);
  printVector("   C", C);
  return C;
}

int main() {
  std::vector<float> A(N);
  std::vector<float> B(N);
  std::vector<float> expected(N);
  for (int i = 0; i < N; ++i) {
    A[i] = static_cast<float>(i);
    B[i] = 10.0f * (i + 1);
    expected[i] = A[i] + B[i];
  }

  printVector("A", A);
  printVector("B", B);
  std::printf("\n");

  const DeviceVectors device = copyInputsToDevice(A, B);
  launchWithoutChecking(device);
  launchAndAskForTheError(device);
  const std::vector<float> C = launchCorrectlyWithCheck(device);
  freeDevice(device);

  const bool ok = C == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 08_error_checking.cu -o 08_error_checking && ./08_error_checking

**What to look at in the output:** Part 1: C still holds the marker -1.0 in every position, so the kernel never ran, and no message appeared. Part 2: `cudaGetLastError()` returns 9, "invalid configuration argument", the same error a real GPU gives. Part 3: the valid launch (1 block of 256 threads) computes C = A + B. Try: set `THREADS_PER_BLOCK` to 1025 and run again: now `CHECK(cudaGetLastError())` in part 3 prints the error with the file and line, and the program stops.

## Step 9 of 12: `09_flatten_3d.cu`

A 3D grid over a 2 × 3 × 4 array: `plane`, `row` and `col` from z, y and x, the 3D guard and the flat index `(plane * height + row) * width + col`. Each thread writes its own coordinates, so the array copied back shows the memory order. The same index function answers exercises 11.4 and 11.5. (slides: *Flattening by hand*, *Exercises (2/2): count the threads, find the index*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 09_flatten_3d.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int WIDTH = 4;
const int HEIGHT = 3;
const int DEPTH = 2;
const int ELEMENTS = WIDTH * HEIGHT * DEPTH;
const int BLOCK_SIDE = 2;
const int NOT_WRITTEN = -1;

__host__ __device__ int flatIndex3d(int plane, int row, int col, int width, int height) {
  return (plane * height + row) * width + col;
}

__host__ __device__ int coordinateCode(int plane, int row, int col) {
  return plane * 100 + row * 10 + col;
}

__global__ void writeCoordinates(int* P, int width, int height, int depth) {
  int plane = blockIdx.z * blockDim.z + threadIdx.z;
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if (plane < depth && row < height && col < width) {
    P[flatIndex3d(plane, row, col, width, height)] = coordinateCode(plane, row, col);
  }
}

static int blocksToCover(int n, int blockSide) {
  return (n + blockSide - 1) / blockSide;
}

static std::vector<int> writeCoordinatesOnDevice(dim3 grid, dim3 block) {
  const size_t bytes = ELEMENTS * sizeof(int);
  std::vector<int> P(ELEMENTS, NOT_WRITTEN);

  int* P_d;
  CHECK(cudaMalloc(&P_d, bytes));
  CHECK(cudaMemcpy(P_d, P.data(), bytes, cudaMemcpyHostToDevice));

  launch(writeCoordinates, grid, block, P_d, WIDTH, HEIGHT, DEPTH);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(P.data(), P_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(P_d));
  return P;
}

static std::vector<int> coordinatesInMemoryOrderOnCpu() {
  std::vector<int> P(ELEMENTS);
  for (int flat = 0; flat < ELEMENTS; ++flat) {
    const int plane = flat / (WIDTH * HEIGHT);
    const int row = flat / WIDTH % HEIGHT;
    const int col = flat % WIDTH;
    P[flat] = coordinateCode(plane, row, col);
  }
  return P;
}

static void printMemory(const std::vector<int>& P) {
  std::printf("memory, one line, as (plane row col):\n");
  for (int flat = 0; flat < ELEMENTS; ++flat) {
    if (flat > 0 && flat % (WIDTH * HEIGHT) == 0) {
      std::printf(" |");
    }
    std::printf(" %03d", P[flat]);
  }
  std::printf("\n");
}

static void printPlanes(const std::vector<int>& P) {
  for (int plane = 0; plane < DEPTH; ++plane) {
    std::printf("plane %d, flat index of each (row, col):\n", plane);
    for (int row = 0; row < HEIGHT; ++row) {
      for (int col = 0; col < WIDTH; ++col) {
        const int flat = flatIndex3d(plane, row, col, WIDTH, HEIGHT);
        std::printf(" %2d:%03d", flat, P[flat]);
      }
      std::printf("\n");
    }
  }
}

static void printExerciseIndices() {
  const int mapWidth = 300;
  const int mapHeight = 200;
  std::printf("\nexercises 11.4 and 11.5, a %d x %d map:\n", mapWidth, mapHeight);
  std::printf("  row 15, col 40, row-major:    15 * %d + 40 = %d\n",
              mapWidth,
              flatIndex3d(0, 15, 40, mapWidth, mapHeight));
  std::printf(
      "  row 15, col 40, column-major: 40 * %d + 15 = %d\n", mapHeight, 40 * mapHeight + 15);
  std::printf("  x 40, y 15, z 3, row-major:   (3 * %d + 15) * %d + 40 = %d\n",
              mapHeight,
              mapWidth,
              flatIndex3d(3, 15, 40, mapWidth, mapHeight));
}

int main() {
  const dim3 block(BLOCK_SIDE, BLOCK_SIDE, BLOCK_SIDE);
  const dim3 grid(blocksToCover(WIDTH, BLOCK_SIDE),
                  blocksToCover(HEIGHT, BLOCK_SIDE),
                  blocksToCover(DEPTH, BLOCK_SIDE));
  const int launched = grid.x * grid.y * grid.z * BLOCK_SIDE * BLOCK_SIDE * BLOCK_SIDE;

  std::printf("%d planes x %d rows x %d columns, blocks of %d x %d x %d, grid of %u x %u x %u\n",
              DEPTH,
              HEIGHT,
              WIDTH,
              BLOCK_SIDE,
              BLOCK_SIDE,
              BLOCK_SIDE,
              grid.x,
              grid.y,
              grid.z);
  std::printf("%d threads launched for %d elements, %d stop at the 3D guard\n",
              launched,
              ELEMENTS,
              launched - ELEMENTS);
  std::printf(
      "thread (plane, row, col) writes plane*100 + row*10 + col at (plane*%d + row)*%d + col\n\n",
      HEIGHT,
      WIDTH);

  const std::vector<int> P = writeCoordinatesOnDevice(grid, block);
  const std::vector<int> expected = coordinatesInMemoryOrderOnCpu();
  printMemory(P);
  printPlanes(P);
  printExerciseIndices();

  const bool ok = P == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 09_flatten_3d.cu -o 09_flatten_3d && ./09_flatten_3d

**What to look at in the output:** The memory line reads 000 001 002 003 010 … 123: the column changes fastest, then the row, then the plane, and plane 1 starts at index 12 = 1 × 3 × 4. 32 threads are launched for 24 elements and 8 stop at the guard: they have row 3, which does not exist. The last lines are the exercise answers: 4540 row-major, 8015 column-major and 184540 in 3D.

## Step 10 of 12: `10_count_threads.cu`

The `scaleKernel` of exercise 11.3 on its 200 × 120 matrix with blocks of 32 × 8 and the grid `((cols - 1) / 32 + 1, (rows - 1) / 8 + 1)`. Two `atomicAdd` counters count the threads that start and the threads that reach line A. (slide: *Exercises (2/2): count the threads, find the index*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 10_count_threads.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const unsigned ROWS = 200;
const unsigned COLS = 120;
const unsigned ELEMENTS = ROWS * COLS;
const unsigned BLOCK_X = 32;
const unsigned BLOCK_Y = 8;
const unsigned CORNER = 3;

struct ThreadCounts {
  unsigned launched;
  unsigned ranLineA;
};

__global__ void scaleKernel(
    const float* in, float* out, unsigned rows, unsigned cols, ThreadCounts* counts) {
  atomicAdd(&counts->launched, 1u);
  unsigned row = blockIdx.y * blockDim.y + threadIdx.y;
  unsigned col = blockIdx.x * blockDim.x + threadIdx.x;
  if (row < rows && col < cols) {
    out[row * cols + col] = in[row * cols + col] * 1.5f;
    atomicAdd(&counts->ranLineA, 1u);
  }
}

static ThreadCounts scale(const float* in_d, float* out_d, dim3 grid, dim3 block) {
  ThreadCounts counts{0, 0};
  ThreadCounts* counts_d;
  CHECK(cudaMalloc(&counts_d, sizeof(ThreadCounts)));
  CHECK(cudaMemcpy(counts_d, &counts, sizeof(ThreadCounts), cudaMemcpyHostToDevice));

  launch(scaleKernel, grid, block, in_d, out_d, ROWS, COLS, counts_d);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(&counts, counts_d, sizeof(ThreadCounts), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(counts_d));
  return counts;
}

static std::vector<float> scaleOnCpu(const std::vector<float>& in) {
  std::vector<float> out(ELEMENTS);
  for (unsigned i = 0; i < ELEMENTS; ++i) {
    out[i] = in[i] * 1.5f;
  }
  return out;
}

static void printCorner(const char* title, const std::vector<float>& matrix) {
  std::printf("%s\n", title);
  for (unsigned row = 0; row < CORNER; ++row) {
    for (unsigned col = 0; col < CORNER; ++col) {
      std::printf(" %5.1f", matrix[row * COLS + col]);
    }
    std::printf("  ...\n");
  }
}

int main() {
  std::vector<float> in(ELEMENTS);
  for (unsigned i = 0; i < ELEMENTS; ++i) {
    in[i] = static_cast<float>(i % 10);
  }

  const size_t bytes = ELEMENTS * sizeof(float);
  float* in_d;
  float* out_d;
  CHECK(cudaMalloc(&in_d, bytes));
  CHECK(cudaMalloc(&out_d, bytes));
  CHECK(cudaMemcpy(in_d, in.data(), bytes, cudaMemcpyHostToDevice));

  const dim3 block(BLOCK_X, BLOCK_Y);
  const dim3 grid((COLS - 1) / BLOCK_X + 1, (ROWS - 1) / BLOCK_Y + 1);
  const ThreadCounts counts = scale(in_d, out_d, grid, block);

  std::vector<float> out(ELEMENTS);
  CHECK(cudaMemcpy(out.data(), out_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(in_d));
  CHECK(cudaFree(out_d));

  const unsigned threadsPerBlock = block.x * block.y;
  const unsigned blocks = grid.x * grid.y;
  std::printf("rows = %u, cols = %u, block (%u, %u), grid ((%u - 1) / %u + 1, (%u - 1) / %u + 1)\n",
              ROWS,
              COLS,
              BLOCK_X,
              BLOCK_Y,
              COLS,
              BLOCK_X,
              ROWS,
              BLOCK_Y);
  std::printf("(a) threads per block:     %u x %u = %u\n", block.x, block.y, threadsPerBlock);
  std::printf("(b) blocks in the grid:    %u x %u = %u\n", grid.x, grid.y, blocks);
  std::printf("(c) threads in the grid:   %u x %u = %u, counted by the kernel: %u\n",
              blocks,
              threadsPerBlock,
              blocks * threadsPerBlock,
              counts.launched);
  std::printf("(d) threads on line A:     %u x %u = %u, counted by the kernel: %u\n\n",
              ROWS,
              COLS,
              ELEMENTS,
              counts.ranLineA);
  printCorner("in, top-left corner:", in);
  printCorner("out = in * 1.5, top-left corner:", out);

  const bool ok = out == scaleOnCpu(in) && counts.launched == blocks * threadsPerBlock &&
                  counts.ranLineA == ELEMENTS;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 10_count_threads.cu -o 10_count_threads && ./10_count_threads

**What to look at in the output:** The four answers of 11.3, each worked out and then counted by the kernel: 256 threads per block, 4 × 25 = 100 blocks, 25600 threads, and 24000 of them on line A, one per element; the other 1600 fail the guard. The corner shows out = in × 1.5.

## Step 11 of 12: `11_matmul_row_col.cu`

The two kernels of exercise 11.1: `matMulRowKernel`, where each thread computes a whole row of P, and `matMulColKernel`, a whole column, both launched with `<<<(width + 255) / 256, 256>>>` on the 4 × 4 matrices of step 7. (slide: *Solution 11.1 · a row or a column per thread*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 11_matmul_row_col.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int WIDTH = 4;
const int ELEMENTS = WIDTH * WIDTH;
const int THREADS_PER_BLOCK = 256;
const int NOT_COMPUTED = -1;

using Matrix = std::vector<float>;

__global__ void matMulRowKernel(const float* M, const float* N, float* P, int width) {
  int row = blockIdx.x * blockDim.x + threadIdx.x;
  if (row < width) {
    for (int col = 0; col < width; ++col) {
      float sum = 0;
      for (int k = 0; k < width; ++k) {
        sum += M[row * width + k] * N[k * width + col];
      }
      P[row * width + col] = sum;
    }
  }
}

__global__ void matMulColKernel(const float* M, const float* N, float* P, int width) {
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  if (col < width) {
    for (int row = 0; row < width; ++row) {
      float sum = 0;
      for (int k = 0; k < width; ++k) {
        sum += M[row * width + k] * N[k * width + col];
      }
      P[row * width + col] = sum;
    }
  }
}

using MatMulKernel = void (*)(const float*, const float*, float*, int);

static Matrix matMulOnDevice(MatMulKernel kernel, const Matrix& M, const Matrix& N) {
  const size_t bytes = ELEMENTS * sizeof(float);
  Matrix P(ELEMENTS, NOT_COMPUTED);

  float* M_d;
  float* N_d;
  float* P_d;
  CHECK(cudaMalloc(&M_d, bytes));
  CHECK(cudaMalloc(&N_d, bytes));
  CHECK(cudaMalloc(&P_d, bytes));
  CHECK(cudaMemcpy(M_d, M.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(N_d, N.data(), bytes, cudaMemcpyHostToDevice));

  const int blocks = (WIDTH + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
  launch(kernel, dim3(blocks), dim3(THREADS_PER_BLOCK), M_d, N_d, P_d, WIDTH);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(P.data(), P_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(M_d));
  CHECK(cudaFree(N_d));
  CHECK(cudaFree(P_d));
  return P;
}

static Matrix matMulOnCpu(const Matrix& M, const Matrix& N) {
  Matrix P(ELEMENTS, 0.0f);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      float sum = 0;
      for (int k = 0; k < WIDTH; ++k) {
        sum += M[row * WIDTH + k] * N[k * WIDTH + col];
      }
      P[row * WIDTH + col] = sum;
    }
  }
  return P;
}

static void printMatrix(const char* name, const Matrix& matrix) {
  std::printf("%s =\n", name);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      std::printf(" %6.1f", matrix[row * WIDTH + col]);
    }
    std::printf("\n");
  }
}

int main() {
  Matrix M(ELEMENTS);
  Matrix N(ELEMENTS);
  for (int i = 0; i < ELEMENTS; ++i) {
    M[i] = static_cast<float>(i + 1);
    N[i] = static_cast<float>((i % WIDTH + i / WIDTH) % 3);
  }

  const int blocks = (WIDTH + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
  std::printf("%d x %d matrices, launch <<<(%d + %d) / %d, %d>>> = %d block of %d threads\n",
              WIDTH,
              WIDTH,
              WIDTH,
              THREADS_PER_BLOCK - 1,
              THREADS_PER_BLOCK,
              THREADS_PER_BLOCK,
              blocks,
              THREADS_PER_BLOCK);
  std::printf("only %d threads work: one per row (or column) of P, each computes %d elements\n",
              WIDTH,
              WIDTH);
  std::printf("for width 1000: 1000 threads, against 1000000 with one element per thread\n\n");
  printMatrix("M", M);
  printMatrix("N", N);

  const Matrix expected = matMulOnCpu(M, N);
  const Matrix byRow = matMulOnDevice(matMulRowKernel, M, N);
  const Matrix byCol = matMulOnDevice(matMulColKernel, M, N);
  printMatrix("P, one thread per row (matMulRowKernel)", byRow);
  printMatrix("P, one thread per column (matMulColKernel)", byCol);

  const bool ok = byRow == expected && byCol == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 11_matmul_row_col.cu -o 11_matmul_row_col && ./11_matmul_row_col

**What to look at in the output:** Both P matrices equal the one of step 7 (P[0][0] = 8), but only 4 of the 256 threads do any work, and each one computes 4 elements instead of 1. For width 1000 that is 1000 busy threads against 1000000 in step 7.

## Step 12 of 12: `12_matvec.cu`

The matrix-vector product of exercise 11.2: `matVecKernel`, one thread per output element, and the host function `matVec(a, B, c, N)` that allocates, copies B and c, launches, copies a back and frees. (slide: *Solution 11.2 · matrix-vector product*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 12_matvec.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int SIZE = 5;
const int THREADS_PER_BLOCK = 256;

__global__ void matVecKernel(float* a, const float* B, const float* c, int N) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < N) {
    float sum = 0;
    for (int j = 0; j < N; ++j) {
      sum += B[i * N + j] * c[j];
    }
    a[i] = sum;
  }
}

static void matVec(float* a, const float* B, const float* c, int N) {
  const size_t vec = N * sizeof(float);
  const size_t mat = (size_t)N * N * sizeof(float);

  float* a_d;
  float* B_d;
  float* c_d;
  CHECK(cudaMalloc(&a_d, vec));
  CHECK(cudaMalloc(&B_d, mat));
  CHECK(cudaMalloc(&c_d, vec));
  CHECK(cudaMemcpy(B_d, B, mat, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(c_d, c, vec, cudaMemcpyHostToDevice));

  dim3 block(THREADS_PER_BLOCK);
  dim3 grid((N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK);
  launch(matVecKernel, grid, block, a_d, B_d, c_d, N);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(a, a_d, vec, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(a_d));
  CHECK(cudaFree(B_d));
  CHECK(cudaFree(c_d));
}

static std::vector<float> matVecOnCpu(const std::vector<float>& B, const std::vector<float>& c) {
  std::vector<float> a(SIZE, 0.0f);
  for (int i = 0; i < SIZE; ++i) {
    for (int j = 0; j < SIZE; ++j) {
      a[i] += B[i * SIZE + j] * c[j];
    }
  }
  return a;
}

int main() {
  std::vector<float> B(SIZE * SIZE);
  std::vector<float> c(SIZE);
  for (int i = 0; i < SIZE; ++i) {
    c[i] = static_cast<float>(i + 1);
    for (int j = 0; j < SIZE; ++j) {
      B[i * SIZE + j] = static_cast<float>(j <= i ? 1 : 0);
    }
  }

  std::vector<float> a(SIZE, 0.0f);
  matVec(a.data(), B.data(), c.data(), SIZE);
  const std::vector<float> expected = matVecOnCpu(B, c);

  std::printf("N = %d, %d block of %d threads, thread i computes a[i] = row i of B times c\n\n",
              SIZE,
              (SIZE + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK,
              THREADS_PER_BLOCK);
  for (int i = 0; i < SIZE; ++i) {
    std::printf("%s", i == SIZE / 2 ? "B =" : "   ");
    for (int j = 0; j < SIZE; ++j) {
      std::printf(" %3.0f", B[i * SIZE + j]);
    }
    std::printf("   %s %3.0f", i == SIZE / 2 ? "c =" : "   ", c[i]);
    std::printf("   %s %5.1f\n", i == SIZE / 2 ? "a =" : "   ", a[i]);
  }

  const bool ok = a == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 12_matvec.cu -o 12_matvec && ./12_matvec

**What to look at in the output:** B is lower triangular with ones, so row i of B picks the first i + 1 values of c and a holds the running sums 1, 3, 6, 10, 15. Thread 2 walks row 2 of B, which starts at B[2 × 5].

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/01_first_kernels/README.md`.

**Next:** open `ec2/02_gpu_architecture/colab_1_step_by_step.ipynb`.